# <center> Evaluations </center>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import warnings
from sklearn.metrics import (precision_recall_curve, precision_score,
                             recall_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix)
warnings.filterwarnings("ignore")

SEED = 42

# Carregar dados
X = pd.read_csv("../data/processed/X_features.csv")
y = pd.read_csv("../data/processed/y_target.csv").squeeze()

# Recriar o split com o mesmo SEED — garante exatos mesmos conjuntos
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# Carregar modelo salvo
with open("../models/xgboost_credit.pkl", "rb") as f:
    xgb_model = pickle.load(f)

# Verificar
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")
print(f"Modelo carregado: {type(xgb_model).__name__}")

# Confirmar que o modelo está funcionando
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
print(f"AUC confirmado: {roc_auc_score(y_test, y_proba_xgb):.4f}")

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_xgb)

# Encontrar threshold que maximiza F1
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
idx_melhor = f1_scores.argmax()
melhor_threshold = thresholds[idx_melhor]

print(f"Threshold padrão (0.5):")
y_pred_05 = (y_proba_xgb >= 0.5).astype(int)
print(f"  Precision: {precision_score(y_test, y_pred_05):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_05):.4f}")
print(f"  F1:        {f1_score(y_test, y_pred_05):.4f}")

print(f"\nThreshold ótimo ({melhor_threshold:.4f}):")
y_pred_otimo = (y_proba_xgb >= melhor_threshold).astype(int)
print(f"  Precision: {precision_score(y_test, y_pred_otimo):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_otimo):.4f}")
print(f"  F1:        {f1_score(y_test, y_pred_otimo):.4f}")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresholds, precisions[:-1], "#e74c3c", label="Precision", linewidth=2)
axes[0].plot(thresholds, recalls[:-1], "#2ecc71", label="Recall", linewidth=2)
axes[0].plot(thresholds, f1_scores[:-1], "#3498db", label="F1", linewidth=2)
axes[0].axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Threshold 0.5")
axes[0].axvline(melhor_threshold, color="orange",
                linestyle="--", alpha=0.7,
                label=f"Threshold ótimo ({melhor_threshold:.2f})")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Score")
axes[0].set_title("Precision, Recall e F1 por Threshold", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(recalls, precisions, "#e74c3c", linewidth=2)
axes[1].scatter(recalls[idx_melhor], precisions[idx_melhor],
                color="orange", s=100, zorder=5, label="Threshold ótimo")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Curva Precision-Recall", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# EXPLAINABILITY WITH SHAP

import shap

# Criar explainer para XGBoost
explainer = shap.TreeExplainer(xgb_model)

# Calcular SHAP values para o conjunto de teste
# Usar amostra de 2000 para performance
X_test_sample = X_test.iloc[:2000]
shap_values = explainer.shap_values(X_test_sample)

print(f"Shape dos SHAP values: {shap_values.shape}")
print("SHAP values calculados com sucesso")

# Gráfico 1 — Importância global das features
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_test_sample,
    plot_type="bar",
    show=False
)
plt.title("Importância Global das Features (SHAP)", fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()

# Gráfico 2 — Impacto e direção de cada feature
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_test_sample,
    show=False
)
plt.title("Direção do Impacto das Features (SHAP)", fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

# Explicação individual — cliente de alto risco
# Encontrar cliente com maior probabilidade de inadimplência
idx_alto_risco = y_proba_xgb[:2000].argmax()
print(f"\nCliente de alto risco (índice {idx_alto_risco}):")
print(f"Probabilidade de inadimplência: {y_proba_xgb[idx_alto_risco]:.2%}")
print(f"Inadimplente real: {'Sim' if y_test.iloc[idx_alto_risco] == 1 else 'Não'}")

# Waterfall plot — explica a decisão individual
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[idx_alto_risco],
        base_values=explainer.expected_value,
        data=X_test_sample.iloc[idx_alto_risco],
        feature_names=X_test_sample.columns.tolist()
    ),
    show=False
)
plt.title(f"Explicação Individual — Cliente {idx_alto_risco}", fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# PERFORMANCE EVALUATION BY SEGMENT

df_test = X_test.copy()
df_test["target"] = y_test.values
df_test["proba"] = y_proba_xgb
df_test["pred_otimo"] = (y_proba_xgb >= melhor_threshold).astype(int)

# Segmento 1 — Performance por faixa etária
df_test["age_faixa"] = pd.cut(
    df_test["age"],
    bins=[0, 25, 35, 45, 55, 65, 110],
    labels=["<25", "25-35", "35-45", "45-55", "55-65", "65+"]
)

print("=== AUC por faixa etária ===")
for faixa in df_test["age_faixa"].cat.categories:
    subset = df_test[df_test["age_faixa"] == faixa]
    if subset["target"].sum() > 10:  # mínimo de eventos para calcular AUC
        auc = roc_auc_score(subset["target"], subset["proba"])
        n = len(subset)
        taxa = subset["target"].mean()
        print(f"{faixa:>6}: AUC={auc:.4f} | n={n:,} | inadimplência={taxa:.2%}")

# Segmento 2 — Performance por faixa de renda
df_test["renda_faixa"] = pd.qcut(
    df_test["MonthlyIncome"],
    q=4,
    labels=["Q1 — Baixa", "Q2 — Média-baixa", "Q3 — Média-alta", "Q4 — Alta"]
)

print("\n=== AUC por faixa de renda ===")
for faixa in df_test["renda_faixa"].cat.categories:
    subset = df_test[df_test["renda_faixa"] == faixa]
    if subset["target"].sum() > 10:
        auc = roc_auc_score(subset["target"], subset["proba"])
        n = len(subset)
        taxa = subset["target"].mean()
        print(f"{faixa}: AUC={auc:.4f} | n={n:,} | inadimplência={taxa:.2%}")

# Segmento 3 — Matriz de confusão com threshold ótimo
cm = confusion_matrix(y_test, df_test["pred_otimo"])
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt=",", cmap="Blues",
            xticklabels=["Pred: Adimplente", "Pred: Inadimplente"],
            yticklabels=["Real: Adimplente", "Real: Inadimplente"],
            ax=ax)
ax.set_title(f"Matriz de Confusão — Threshold {melhor_threshold:.2f}",
             fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\n=== Interpretação de negócio ===")
print(f"Verdadeiros Negativos (aprovados corretamente): {tn:,}")
print(f"Falsos Positivos (bons clientes reprovados):    {fp:,}")
print(f"Falsos Negativos (inadimplentes aprovados):     {fn:,}")
print(f"Verdadeiros Positivos (inadimplentes pegos):    {tp:,}")

In [ ]:
# FINANCIAL IMPACT SIMULATION
ticket_medio = 15000        # valor médio do empréstimo (R$)
taxa_perda_inadimplencia = 0.60   # banco recupera 40% via cobrança
receita_por_cliente = 800   # receita média de um bom cliente aprovado

perda_fn = 904 * ticket_medio * taxa_perda_inadimplencia
custo_fp = 1746 * receita_por_cliente

print("=== Impacto Financeiro Estimado (threshold 0.749) ===")
print(f"Inadimplentes que passaram (FN): {904:,}")
print(f"  → Perda estimada: R$ {perda_fn:,.0f}")
print(f"\nBons clientes reprovados (FP): {1.746:,}")
print(f"  → Receita perdida: R$ {custo_fp:,.0f}")
print(f"\nSaldo estimado do modelo: R$ {perda_fn - custo_fp:,.0f}")

In [ ]:
# FINAL EVALUATION - DOCUMENTATION
avaliacao_final = {
    "modelo_escolhido": "XGBoost",
    "threshold_producao": round(float(melhor_threshold), 4),
    "metricas_finais": {
        "auc": 0.8697,
        "ks": 0.5800,
        "f1": round(float(f1_score(y_test, df_test["pred_otimo"])), 4),
        "precision": round(float(precision_score(y_test, df_test["pred_otimo"])), 4),
        "recall": round(float(recall_score(y_test, df_test["pred_otimo"])), 4)
    },
    "matriz_confusao": {
        "verdadeiros_negativos": 26249,
        "falsos_positivos": 1746,
        "falsos_negativos": 904,
        "verdadeiros_positivos": 1101
    },
    "fairness": {
        "auc_por_idade": "variação de 0.022 — aceitável e com direção lógica",
        "auc_por_renda": "variação de 0.019 — sem evidência de discriminação por renda",
        "conclusao": "modelo não apresenta viés discriminatório relevante nos segmentos analisados"
    },
    "features_mais_importantes": [
        "teve_qualquer_atraso",
        "score_risco",
        "total_atrasos",
        "RevolvingUtilizationOfUnsecuredLines",
        "age"
    ]
}

import json
with open("../reports/avaliacao_final.json", "w", encoding="utf-8") as f:
    json.dump(avaliacao_final, f, ensure_ascii=False, indent=2)

print("Avaliação final salva em reports/avaliacao_final.json")
print("\n=== RESUMO EXECUTIVO ===")
print(f"Modelo: XGBoost")
print(f"AUC: 0.8697 — performance forte para credit scoring")
print(f"KS: 0.5800 — acima do benchmark de mercado (0.40)")
print(f"Threshold: {melhor_threshold:.4f} — otimizado para F1")
print(f"Recall: {recall_score(y_test, df_test['pred_otimo']):.2%} — captura mais da metade dos inadimplentes")
print(f"Fairness: sem viés discriminatório identificado")

In [ ]:
# Recarregar X_train pois estamos em novo contexto
X_full = pd.read_csv("../data/processed/X_features.csv")
y_full = pd.read_csv("../data/processed/y_target.csv").squeeze()

X_train, X_test_temp, y_train, y_test_temp = train_test_split(
    X_full, y_full,
    test_size=0.2,
    random_state=SEED,
    stratify=y_full
)

# Recarregar dataset processado para calcular medianas originais
df_processed = pd.read_csv("../data/processed/credit_processed.csv")

# Parâmetros calculados no treino — serão usados na inferência
pipeline_params = {
    "mediana_monthly_income": float(df_processed["MonthlyIncome"].median()),
    "mediana_number_of_dependents": float(df_processed["NumberOfDependents"].median()),
    "p99_revolving": float(df_processed["RevolvingUtilizationOfUnsecuredLines"].quantile(0.99)),
    "p99_monthly_income": float(df_processed["MonthlyIncome"].quantile(0.99)),
    "p99_debt_ratio": float(
        df_processed[df_processed["DebtRatio"] < 2]["DebtRatio"].quantile(0.99)
    ),
    "score_risco_min": float(X_train["score_risco"].min()),
    "score_risco_max": float(X_train["score_risco"].max()),
    "revolving_min": float(X_train["RevolvingUtilizationOfUnsecuredLines"].min()),
    "revolving_max": float(X_train["RevolvingUtilizationOfUnsecuredLines"].max()),
    "atrasos_min": float(X_train["total_atrasos"].min()),
    "atrasos_max": float(X_train["total_atrasos"].max()),
    "threshold_producao": round(float(melhor_threshold), 4),
    "features_ordem": X_train.columns.tolist()
}

with open("../models/pipeline_params.json", "w") as f:
    json.dump(pipeline_params, f, indent=2)

print("Parâmetros salvos em models/pipeline_params.json")
print(f"\nTotal de parâmetros: {len(pipeline_params)}")
print(f"Features na ordem correta: {len(pipeline_params['features_ordem'])}")
print(f"Threshold de produção: {pipeline_params['threshold_producao']}")
print(f"Mediana de renda: R$ {pipeline_params['mediana_monthly_income']:,.0f}")